In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/movie-recommender/pos_neg_records.csv


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import time
import torch.optim as optim
import random # Import random for negative sampling

/kaggle/input/movie-recommender/pos_neg_records.csv


In [3]:
# --- Load Data ---
# We assume the input CSV contains POSITIVE interactions only for this setup.
# If it contains pre-defined negatives, you might need to filter for positives
# or adjust the logic. Let's assume df represents positive interactions.
try:
    # Attempt to load the specific pos_neg_records first
    df= pd.read_csv("/kaggle/input/movie-recommender/pos_neg_records.csv")
    print("Loaded pos_neg_records.csv")
    # For random negative sampling, we only need the positive examples from this file
    # Assuming 'flag' = 1 indicates a positive interaction
    df_positive = df[df['flag'] == 1].copy()
    if df_positive.empty:
        print("Warning: No positive samples found in pos_neg_records.csv based on flag=1. Trying ratings.csv")
        # Fallback or alternative loading if needed
        df_ratings = pd.read_csv("/kaggle/input/movielens-100k-dataset/ml-100k/u.data", sep='\\t', names=['user_id', 'item_id', 'rating', 'timestamp'], engine='python')
        df_info = pd.read_csv("/kaggle/input/movielens-100k-dataset/ml-100k/u.item", sep='|', names=['item_id', 'movie_title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western'], engine='python', encoding='latin-1')
        df_user = pd.read_csv("/kaggle/input/movielens-100k-dataset/ml-100k/u.user", sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code'], engine='python')
        # Preprocess and merge if using raw movielens files
        # Example merge (adjust features as needed):
        df = pd.merge(df_ratings, df_info, on='item_id')
        df = pd.merge(df, df_user, on='user_id')
        # Add gender dummy variables if needed
        df['gender_F'] = (df['gender'] == 'F').astype(int)
        df['gender_M'] = (df['gender'] == 'M').astype(int)
        # Normalize age example
        df['age'] = (df['age'] - df['age'].mean()) / df['age'].std()
        # Assume all interactions in ratings are positive for sampling
        df_positive = df.copy()
        df_positive['flag'] = 1 # Add flag for consistency
        print(f"Loaded and processed raw MovieLens data. Positive samples: {len(df_positive)}")

except FileNotFoundError:
    print("Warning: pos_neg_records.csv not found. Attempting to load raw MovieLens data.")
    # Load raw MovieLens 100k data as a fallback
    df_ratings = pd.read_csv("/kaggle/input/movielens-100k-dataset/ml-100k/u.data", sep='\\t', names=['user_id', 'item_id', 'rating', 'timestamp'], engine='python')
    df_info = pd.read_csv("/kaggle/input/movielens-100k-dataset/ml-100k/u.item", sep='|', names=['item_id', 'movie_title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western'], engine='python', encoding='latin-1')
    df_user = pd.read_csv("/kaggle/input/movielens-100k-dataset/ml-100k/u.user", sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code'], engine='python')
    # Merge
    df = pd.merge(df_ratings, df_info, on='item_id')
    df = pd.merge(df, df_user, on='user_id')
    # Preprocess user features
    df['gender_F'] = (df['gender'] == 'F').astype(int)
    df['gender_M'] = (df['gender'] == 'M').astype(int)
    df['age'] = (df['age'] - df['age'].mean()) / df['age'].std() # Example normalization
    # Assume all interactions in ratings are positive for sampling
    df_positive = df.copy()
    df_positive['flag'] = 1 # Add flag for consistency
    print(f"Loaded and processed raw MovieLens data. Positive samples: {len(df_positive)}")


# --- Constants ---
GENRES = ['unknown','Action','Adventure','Animation','Children','Comedy', 'Crime',
          'Documentary', 'Drama', 'Fantasy','Film_Noir', 'Horror', 'Musical',
          'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western']
TITLE_EMBEDDING_DIM = 16 # Placeholder dimension

Loaded pos_neg_records.csv


In [4]:
class MovieDataset(Dataset):
    """
    Custom PyTorch dataset for movie recommendation with random negative sampling.
    For each positive sample (user, item, 1), it generates a negative sample (user, random_item, 0).
    """
    def __init__(self, data):
        """
        Initializes the dataset with positive interaction data.

        Args:
            data (pd.DataFrame): The dataset containing POSITIVE user-item interactions
                                 and associated features. Must include 'user_id' and 'item_id'.
        """
        super().__init__()
        self.data = data.reset_index(drop=True) # Ensure index is sequential
        self.original_len = len(self.data)

        # --- Mappings ---
        self.user_id_map = {id: i for i, id in enumerate(data['user_id'].unique())}
        self.movie_id_map = {id: i for i, id in enumerate(data['item_id'].unique())}
        self.reverse_user_id_map = {i: id for id, i in self.user_id_map.items()}
        self.reverse_movie_id_map = {i: id for id, i in self.movie_id_map.items()}
        self.num_users = len(self.user_id_map)
        self.num_items = len(self.movie_id_map)

        # --- Data for Negative Sampling ---
        self.all_item_internal_ids = list(self.movie_id_map.values())

        # --- Pre-calculate title embeddings (placeholder) ---
        self.movie_title_embeddings = {}
        if 'movie_title' in data.columns:
            for title in data['movie_title'].unique():
                 # Simple placeholder - in practice use pre-trained embeddings or learn them
                self.movie_title_embeddings[title] = np.random.randn(TITLE_EMBEDDING_DIM)

        # --- Store index of first occurrence for each item_id to fetch features ---
        # This helps quickly find features for a negatively sampled item_id
        item_first_indices = self.data.groupby('item_id').head(1).index
        self.item_id_to_data_index = pd.Series(
            item_first_indices,
            index=self.data.loc[item_first_indices, 'item_id']
        ).to_dict()


    def __len__(self):
        """
        Returns the total number of samples (positive + negative).
        """
        return self.original_len * 2 # One negative sample per positive sample

    def _extract_user_features(self, row):
        """ Helper to extract user features tensor from a data row. """
        # Use .get for robustness against missing columns, provide defaults
        user_features = torch.tensor([
            self.user_id_map[row['user_id']],
            row.get('age', 0), # Use 0 or mean age if missing
            hash(row.get('occupation', 'unknown')) % 20, # Simple hash encoding
            row.get('gender_F', 0),
            row.get('gender_M', 0)
            # Add other user features if available
        ], dtype=torch.float32)
        return user_features

    def _extract_item_features(self, row, item_internal_id):
        """ Helper to extract item features tensor from a data row. """
        # Get movie title embedding
        movie_title = row.get('movie_title', '')
        title_embedding = torch.tensor(
            self.movie_title_embeddings.get(movie_title, np.zeros(TITLE_EMBEDDING_DIM)),
            dtype=torch.float32
        )

        # Extract genres as one-hot encoding
        genre_vector = torch.zeros(len(GENRES), dtype=torch.float32)
        for i, genre in enumerate(GENRES):
            if row.get(genre, 0) == 1:
                genre_vector[i] = 1.0

        # Create item tensor
        item_features = torch.cat([
            torch.tensor([
                item_internal_id, # Use the provided internal ID
                row.get('timestamp', 0) / 1e7,  # Example normalization for timestamp
            ], dtype=torch.float32),
            genre_vector,
            title_embedding
        ])
        return item_features

    def __getitem__(self, index):
        """
        Retrieves a single data sample (positive or negative).

        Args:
            index (int): Index of the sample to fetch (0 to 2*original_len - 1).

        Returns:
            tuple: (user_features, item_features, label_tensor, rating_tensor)
                   label_tensor is 1.0 for positive, 0.0 for negative.
                   rating_tensor is the original rating or 0 for negative.
        """
        if index < self.original_len:
            # --- Positive Sample ---
            is_positive = True
            data_idx = index
            row = self.data.iloc[data_idx]
            item_internal_id = self.movie_id_map[row['item_id']]

        else:
            # --- Negative Sample ---
            is_positive = False
            # Find the corresponding positive sample to get the user
            positive_data_idx = index - self.original_len
            pos_row = self.data.iloc[positive_data_idx]
            user_id = pos_row['user_id']
            positive_item_internal_id = self.movie_id_map[pos_row['item_id']]

            # Sample a negative item
            while True:
                negative_item_internal_id = random.choice(self.all_item_internal_ids)
                if negative_item_internal_id != positive_item_internal_id:
                    # Basic check: ensure it's not the *same* item as the positive one
                    # More advanced: check if user interacted with this item at all (needs precomputation)
                    break

            # Get features for the negative item
            negative_original_item_id = self.reverse_movie_id_map[negative_item_internal_id]
            neg_item_data_idx = self.item_id_to_data_index[negative_original_item_id]
            row = self.data.iloc[neg_item_data_idx] # Get feature row for the negative item
             # Important: Use the user features from the original positive interaction
            user_features = self._extract_user_features(pos_row)
            item_features = self._extract_item_features(row, negative_item_internal_id) # Pass correct internal ID
            label_tensor = torch.tensor([0.0], dtype=torch.float32)
            rating_tensor = torch.tensor([0.0], dtype=torch.float32) # No rating for negative
            return user_features, item_features, label_tensor, rating_tensor


        # --- Process the selected row (positive case) ---
        user_features = self._extract_user_features(row)
        item_features = self._extract_item_features(row, item_internal_id)
        label_tensor = torch.tensor([1.0], dtype=torch.float32) # Positive sample
        rating_tensor = torch.tensor([row.get('rating', 0)], dtype=torch.float32) # Original rating

        return user_features, item_features, label_tensor, rating_tensor


In [5]:
# --- Model Definition (TwoTowerModel) ---
# Note: Adjust input dimensions based on the final feature extraction in MovieDataset
# User features: id(1)+age(1)+occupation(1)+gender(2) = 5 -> After Embedding -> 4 + user_emb_size
# Item features: id(1)+timestamp(1)+genres(19)+title_emb(16) = 37 -> After Embedding -> 36 + item_emb_size

class TwoTowerModel(nn.Module):
    def __init__(self, user_count, item_count, user_feature_size, item_feature_size,
                 user_embedding_dim=16, item_embedding_dim=16, final_embedding_dim=32):
        """
        Initializes the recommendation model.

        Args:
            user_count (int): Total number of unique users.
            item_count (int): Total number of unique items.
            user_feature_size (int): Number of non-ID user features (e.g., age, gender...).
            item_feature_size (int): Number of non-ID item features (e.g., timestamp, genres...).
            user_embedding_dim (int): Size of the user ID embedding vector.
            item_embedding_dim (int): Size of the item ID embedding vector.
            final_embedding_dim (int): Final dimension of both user and item representations.
        """
        super().__init__()

        self.UserEmbedding = nn.Embedding(user_count, user_embedding_dim)
        self.ItemEmbedding = nn.Embedding(item_count, item_embedding_dim)

        # --- User Tower ---
        user_input_dim = user_feature_size + user_embedding_dim
        self.UserModel = nn.Sequential(
            nn.Linear(user_input_dim, 64), # Adjusted layer size
            nn.ReLU(),
            nn.Linear(64, final_embedding_dim) # Output final embedding dim
            #nn.LayerNorm(final_embedding_dim) # Optional normalization
        )

        # --- Item Tower ---
        item_input_dim = item_feature_size + item_embedding_dim
        self.ItemModel = nn.Sequential(
            nn.Linear(item_input_dim, 64), # Adjusted layer size
            nn.ReLU(),
            nn.Linear(64, final_embedding_dim) # Output final embedding dim
            #nn.LayerNorm(final_embedding_dim) # Optional normalization
        )


    def forward(self, user_data, item_data):
        """
        Forward pass: Computes user and item embeddings and calculates interaction score.

        Args:
            user_data (torch.Tensor): User feature tensor [batch_size, user_total_features]
            item_data (torch.Tensor): Item feature tensor [batch_size, item_total_features]

        Returns:
            torch.Tensor: Interaction score (e.g., dot product or classifier output) [batch_size, 1]
        """
        user_vector = self.compute_user_vector(user_data)
        item_vector = self.compute_item_vector(item_data)

        # --- Calculate Score ---
        # Option 1: Dot product (common for two-tower with sampling loss)
        # Ensure vectors are normalized if using cosine similarity interpretation
        # user_vector = F.normalize(user_vector, p=2, dim=1)
        # item_vector = F.normalize(item_vector, p=2, dim=1)
        logits = torch.sum(user_vector * item_vector, dim=1, keepdim=True)

        # Option 2: Pass concatenated vectors through a classifier (if defined)
        # combined = torch.cat([user_vector, item_vector], dim=1)
        # logits = self.ClassifyModel(combined)

        return logits

    def compute_user_vector(self, user_data):
        """ Compute vector representation for user(s). """
        if user_data.dim() == 1: # Handle single instance case
            user_data = user_data.unsqueeze(0)

        user_id = user_data[:, 0].long()
        user_features = user_data[:, 1:]
        user_embedding = self.UserEmbedding(user_id)
        user_input = torch.cat([user_features, user_embedding], dim=1)
        user_vector = self.UserModel(user_input)
        return user_vector

    def compute_item_vector(self, item_data):
        """ Compute vector representation for item(s). """
        if item_data.dim() == 1: # Handle single instance case
            item_data = item_data.unsqueeze(0)

        item_id = item_data[:, 0].long()
        item_features = item_data[:, 1:]
        item_embedding = self.ItemEmbedding(item_id)
        item_input = torch.cat([item_features, item_embedding], dim=1)
        item_vector = self.ItemModel(item_input)
        return item_vector


# --- Training Function ---
def train_model(model, train_loader, test_loader, epochs=5, device='cpu'):
    start_time = time.time()
    criterion = nn.BCEWithLogitsLoss() # Suitable for binary classification (pos/neg samples)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.to(device)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        for i, (user_features, item_features, labels, _) in enumerate(train_loader):
            user_features = user_features.to(device)
            item_features = item_features.to(device)
            labels = labels.to(device) # Labels are now 0 or 1

            optimizer.zero_grad()
            logits = model(user_features, item_features) # Calculate interaction score
            loss = criterion(logits, labels) # Compare score with 0/1 label
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

            if (i + 1) % 50 == 0: # Print progress every 50 batches
                 print(f"Epoch [{epoch+1}/{epochs}], Batch [{i+1}/{len(train_loader)}], Avg Loss: {epoch_loss / num_batches:.4f}")

        avg_epoch_loss = epoch_loss / num_batches
        print(f"--- Epoch {epoch+1} Completed --- Avg Train Loss: {avg_epoch_loss:.4f} ---")

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        val_batches = 0
        with torch.no_grad():
            for user_features, item_features, labels, _ in test_loader:
                user_features = user_features.to(device)
                item_features = item_features.to(device)
                labels = labels.to(device)

                logits = model(user_features, item_features)
                loss = criterion(logits, labels)
                val_loss += loss.item()

                # Calculate accuracy
                preds = (torch.sigmoid(logits) > 0.5).float()
                accuracy = (preds == labels).float().mean()
                val_acc += accuracy.item()
                val_batches += 1

        avg_val_loss = val_loss / val_batches
        avg_val_acc = val_acc / val_batches
        print(f"Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {avg_val_acc:.4f}")

    end_time = time.time()
    print(f"Total Training Time: {end_time - start_time:.2f} seconds")
    return model


# --- Recommendation Function (Modified) ---
def recommend_items(model, user_features_single, item_dataset, top_n=10, device='cpu'):
    """
    Recommend top-N unique items for a user using the trained model.

    Args:
        model: Trained TwoTowerModel.
        user_features_single (torch.Tensor): Feature tensor for the target user (single instance).
        item_dataset (MovieDataset): The dataset instance to access item info.
        top_n (int): Number of recommendations to return.
        device: Device for computation.

    Returns:
        list: Top-N recommended original item_ids.
    """
    model.eval()
    model.to(device)
    user_features_single = user_features_single.to(device)

    # Compute user representation (ensure batch dim)
    with torch.no_grad():
        user_vector = model.compute_user_vector(user_features_single.unsqueeze(0)) # Shape [1, emb_dim]

    item_scores = {} # {original_item_id: score}

    with torch.no_grad():
        # Iterate through all *unique* items in the dataset
        for original_item_id, internal_item_id in item_dataset.movie_id_map.items():
            # Fetch features for this unique item
            try:
                item_data_idx = item_dataset.item_id_to_data_index[original_item_id]
                item_row = item_dataset.data.iloc[item_data_idx]
                item_features = item_dataset._extract_item_features(item_row, internal_item_id)
                item_features = item_features.to(device)

                # Compute item vector (ensure batch dim)
                item_vector = model.compute_item_vector(item_features.unsqueeze(0)) # Shape [1, emb_dim]

                # Calculate score (dot product)
                score = torch.sum(user_vector * item_vector).item()
                item_scores[original_item_id] = score

            except KeyError:
                print(f"Warning: Could not find features for item_id {original_item_id} during recommendation.")
                continue # Skip if item features aren't found

    # Sort items by score
    sorted_items = sorted(item_scores.items(), key=lambda item: item[1], reverse=True)

    # Get top-N original item IDs
    top_original_ids = [item_id for item_id, score in sorted_items[:top_n]]

    return top_original_ids


In [6]:
# --- Main Execution ---

# 1. Create Dataset (using only positive interactions)
dataset = MovieDataset(df_positive)

# 2. Split into train and test sets (indices based on original positive data length)
# We split indices from 0 to original_len-1
positive_indices = list(range(dataset.original_len))
train_indices_pos, test_indices_pos = train_test_split(positive_indices, test_size=0.2, random_state=42)

# Create the full train/test indices including negative counterparts
train_indices = train_indices_pos + [i + dataset.original_len for i in train_indices_pos]
test_indices = test_indices_pos + [i + dataset.original_len for i in test_indices_pos]

# Create Subset datasets using these indices
train_dataset = torch.utils.data.Subset(dataset, train_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)


# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2) # Increased batch size
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=2) # Increased batch size

# 4. Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 5. Initialize model
# Get feature sizes from a sample (ensure correct dimensions)
sample_user_feat, sample_item_feat, _, _ = dataset[0]
user_non_id_feature_size = sample_user_feat.shape[0] - 1
item_non_id_feature_size = sample_item_feat.shape[0] - 1

model = TwoTowerModel(
    user_count=dataset.num_users,
    item_count=dataset.num_items,
    user_feature_size=user_non_id_feature_size,
    item_feature_size=item_non_id_feature_size,
    user_embedding_dim=32, # Increased embedding size
    item_embedding_dim=32, # Increased embedding size
    final_embedding_dim=64  # Increased final embedding size
).to(device)

print("Model initialized:")
print(model)


# 6. Train model
trained_model = train_model(model, train_loader, test_loader, epochs=5, device=device) # Reduced epochs for demo

# 7. Save model
torch.save(trained_model.state_dict(), 'two_tower_neg_sampling_model.pth')
print("Model training completed and saved!")

Using device: cpu
Model initialized:
TwoTowerModel(
  (UserEmbedding): Embedding(943, 32)
  (ItemEmbedding): Embedding(1682, 32)
  (UserModel): Sequential(
    (0): Linear(in_features=36, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (ItemModel): Sequential(
    (0): Linear(in_features=68, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
)
Epoch [1/5], Batch [50/625], Avg Loss: 0.7049
Epoch [1/5], Batch [100/625], Avg Loss: 0.6903
Epoch [1/5], Batch [150/625], Avg Loss: 0.6783
Epoch [1/5], Batch [200/625], Avg Loss: 0.6668
Epoch [1/5], Batch [250/625], Avg Loss: 0.6568
Epoch [1/5], Batch [300/625], Avg Loss: 0.6459
Epoch [1/5], Batch [350/625], Avg Loss: 0.6362
Epoch [1/5], Batch [400/625], Avg Loss: 0.6289
Epoch [1/5], Batch [450/625], Avg Loss: 0.6221
Epoch [1/5], Batch [500/625], Avg Loss: 0.6166
Epoch [1/5], Batch [550/625], Avg Loss: 0.6112
Epoch [1/5], Batch 

In [7]:

# --- Example Recommendation ---
# Get an example user's features (make sure it's just the features, not the whole tuple)
# Choose an index corresponding to a positive sample in the *original* dataset
example_user_original_index = 150 # Example index
example_user_tensor, _, _, _ = dataset[example_user_original_index] # Get features

# Get recommendations (original item IDs)
top_recommended_ids = recommend_items(trained_model, example_user_tensor, dataset, top_n=10, device=device)

print(f"\nTop recommendations for example user (Original IDs): {top_recommended_ids}\n")

# --- Format recommendations with titles and genres ---
recommendations_formatted = []
for movie_id in top_recommended_ids:
    # Find movie details in the original DataFrame (df_positive used for dataset creation)
    movie_row = df_positive[df_positive['item_id'] == movie_id]
    if movie_row.empty:
        print(f"Warning: Could not find details for movie_id {movie_id} in source DataFrame.")
        # Try looking in the full df if it exists and differs
        if 'df' in locals() and df is not df_positive:
             movie_row = df[df['item_id'] == movie_id]
        if movie_row.empty:
             recommendations_formatted.append({'movie_id': int(movie_id), 'title': f"Movie {movie_id} (Not Found)", 'genres': []})
             continue

    movie_row = movie_row.iloc[0] # Take the first instance if duplicates exist
    movie_title = movie_row.get('movie_title', f"Movie {movie_id}")

    # Get genres
    genres = []
    for genre in GENRES:
        if movie_row.get(genre, 0) == 1:
            genres.append(genre)

    recommendations_formatted.append({
        'movie_id': int(movie_id),
        'title': movie_title,
        'genres': genres
    })

print("Formatted Recommendations:")
for rec in recommendations_formatted:
    print(rec)


Top recommendations for example user (Original IDs): [49, 256, 293, 126, 287, 120, 567, 172, 167, 312]

Formatted Recommendations:
{'movie_id': 49, 'title': 'star wars (1977)', 'genres': ['Action', 'Adventure', 'Romance', 'Sci_Fi', 'War']}
{'movie_id': 256, 'title': 'men in black (1997)', 'genres': ['Action', 'Adventure', 'Comedy', 'Sci_Fi']}
{'movie_id': 293, 'title': 'liar liar (1997)', 'genres': ['Comedy']}
{'movie_id': 126, 'title': 'godfather the (1972)', 'genres': ['Action', 'Crime', 'Drama']}
{'movie_id': 287, 'title': 'scream (1996)', 'genres': ['Horror', 'Thriller']}
{'movie_id': 120, 'title': 'independence day (id4) (1996)', 'genres': ['Action', 'Sci_Fi', 'War']}
{'movie_id': 567, 'title': 'speed (1994)', 'genres': ['Action', 'Romance', 'Thriller']}
{'movie_id': 172, 'title': 'princess bride the (1987)', 'genres': ['Action', 'Adventure', 'Comedy', 'Romance']}
{'movie_id': 167, 'title': 'monty python and the holy grail (1974)', 'genres': ['Comedy']}
{'movie_id': 312, 'title':